# 3_6 — Абляции DPI-Flow (P1, component-contribution)

Отдельный ноутбук абляций. Каждая 1:1 отключает один компонент вклада; обучение/оценка на объектном (leakage-free) фолде — тот же протокол, что P0. Метрики **по 3 состояниям** (liq/stab/nostab + worst).

Варианты: full, wo_conformal, gaussian_posterior, wo_ode, wo_monotone, wo_risk_softauc, wo_censored_nliq, miss_vs, miss_grainsize. Логика — в `liquefaction_ai.evaluation.ablation_study`.

In [1]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')):
    REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src'))
os.chdir(REPO)
import numpy as np, pandas as pd
TABLES = os.path.join(REPO, 'results', 'analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB = os.path.join(REPO, 'results', 'tables'); os.makedirs(PUB, exist_ok=True)
FIGS = os.path.join(REPO, 'results', 'analysis_figs'); os.makedirs(FIGS, exist_ok=True)
print('REPO =', REPO)

REPO = /Users/nikita/Desktop/projects/liquefaction-ai


In [2]:
import json, torch
from liquefaction_ai import load_population_artifact
from liquefaction_ai.evaluation.ablation_study import run_ablation_fold, aggregate_ablations
device = torch.device('cpu'); QUICK = False
pop, config = load_population_artifact('data/dataset')
meta = pop['meta']
feat_names = json.load(open('data/dataset/feature_names.json'))['static_feature_names']
from liquefaction_ai.evaluation.cross_validation import build_folds
folds = build_folds(meta, config, seed=42, loo=False, n_splits=5)
print('folds:', len(folds))

folds: 5


## Прогон абляций (по умолчанию фолд 0; для CI — по всем фолдам)

In [3]:
FOLDS = []   # [] → все 5 фолдов (n_splits=5); центральный architectural claim — по всем фолдам
use_folds = FOLDS if FOLDS else range(len(folds))
rows = []
for k in use_folds:
    rows.append(run_ablation_fold(pop, config, folds[k], k, feat_names, device, quick=QUICK))
abl = pd.concat(rows, ignore_index=True)
abl.to_csv(os.path.join(TABLES, 'ablations.csv'), index=False)
abl[['fold','ablation','AUPRC','ECE','Coverage_90','Traj_RMSE_worst','Physics_Violation_Rate','N_liq_logMAE']].round(4)

,fold,ablation,AUPRC,ECE,Coverage_90,Traj_RMSE_worst,Physics_Violation_Rate,N_liq_logMAE
0,0,full,0.9704,0.0926,0.7667,0.1817,0.0000,0.2896
1,0,wo_conformal,0.9704,0.0926,0.6008,0.1817,0.0000,0.2896
2,0,gaussian_posterior,0.9650,0.1209,0.8099,0.1310,0.0000,0.2884
3,0,wo_ode,0.9719,0.1453,0.8053,0.0894,0.0000,0.7042
4,0,blackbox_cummax,0.9719,0.1453,0.8053,0.0894,0.0000,0.7042
5,0,blackbox_raw,0.9884,0.1330,0.8033,0.0698,0.3909,0.6739
6,0,wo_monotone,0.9725,0.0906,0.7462,0.2073,0.0619,0.2816
7,0,wo_risk_softauc,0.9678,0.1162,0.7503,0.1586,0.0000,0.2890
8,0,wo_censored_nliq,0.9713,0.1047,0.7735,0.1752,0.0000,0.2861
9,0,miss_vs,0.9609,0.0953,0.8006,0.1362,0.0000,0.2946


In [4]:
summary = aggregate_ablations(abl)
summary.to_csv(os.path.join(TABLES, 'ablations_summary.csv'), index=False)
summary[[c for c in ['ablation','n_folds','AUPRC_mean','ECE_mean','Traj_RMSE_worst_mean','Physics_Violation_Rate_mean','N_liq_logMAE_mean'] if c in summary.columns]]

,ablation,n_folds,AUPRC_mean,ECE_mean,Traj_RMSE_worst_mean,Physics_Violation_Rate_mean,N_liq_logMAE_mean
0,blackbox_cummax,3,0.9859,0.0798,0.0723,0.0000,0.5968
1,blackbox_raw,3,0.9853,0.0759,0.0673,0.2474,0.6678
2,full,3,0.9662,0.0693,0.1617,0.0014,0.3407
3,gaussian_posterior,3,0.9695,0.0737,0.1487,0.0028,0.3381
4,miss_grainsize,3,0.9830,0.0585,0.1686,0.0014,0.3416
5,miss_vs,3,0.9639,0.0634,0.1448,0.0014,0.3401
6,no_aux,3,0.9656,0.0675,0.1650,0.0014,0.3377
7,no_prefix,3,0.8710,0.1902,0.1888,0.0000,0.3749
8,wo_censored_nliq,3,0.9721,0.0689,0.1618,0.0014,0.3376
9,wo_conformal,3,0.9662,0.0693,0.1617,0.0014,0.3407


## Ablation figures (component contribution, English)

In [5]:
from liquefaction_ai.evaluation import ablation_bars
from IPython.display import display
display(ablation_bars(summary, metric='Traj_RMSE_worst', higher_better=False, out_dir=FIGS))
display(ablation_bars(summary, metric='AUPRC', higher_better=True, out_dir=FIGS))

<Figure size 720x790 with 1 Axes>

<Figure size 720x790 with 1 Axes>

## Prefix-length sensitivity (п.6)

Ре-материализуйте `data/dataset` с разным `config.prefix_len` (ноутбуки 1_x), затем для каждого K запустите этот ноутбук с `only='full'` и сравните метрики онсета/траектории vs длина префикса.